In [4]:
import numpy as np
from numpy.typing import NDArray

X: NDArray[np.uint16] = np.loadtxt(
    "Project_Files/X.csv", delimiter=",", dtype=str
).astype(np.uint16)
labels: NDArray[np.uint16] = np.loadtxt(
    "Project_Files/labels.csv", delimiter=",", dtype=str
).astype(np.uint16)
sigmoid: NDArray[np.uint16] = np.loadtxt(
    "Project_Files/sigmoid.csv", delimiter=",", dtype=str
).astype(np.uint16)
w_hid: NDArray[np.uint16] = np.loadtxt(
    "Project_Files/w_hid.csv", delimiter=",", dtype=str
).astype(np.uint16)
w_out: NDArray[np.uint16] = np.loadtxt(
    "Project_Files/w_out.csv", delimiter=",", dtype=str
).astype(np.uint16)

print(f"X shape: {X.shape}")
print(f"labels shape: {labels.shape}")
print(f"sigmoid shape: {sigmoid.shape}")
print(f"w_hid shape: {w_hid.shape}")
print(f"w_out shape: {w_out.shape}")


Xb: NDArray[np.uint16] = np.hstack((np.ones((64, 1), np.uint16) * 255, X))
# print(Xb)
print(f"Xb shape: {Xb.shape}")

# NOTE: matmul #1: (64,8)*(8,2)
# === two (64,8) * (8,1)
N: NDArray[np.uint16] = Xb @ w_hid // 256
print(N)
print(f"N shape: {N.shape}")


def sig(a):
    return sigmoid[a]


vectorized_sig = np.vectorize(sig)

Na: NDArray[np.uint16] = vectorized_sig(N)
# print(Na)
print(f"Na shape: {Na.shape}")

Nb: NDArray[np.uint16] = np.hstack((np.ones((64, 1), np.uint16) * 255, Na))
# print(Nb)
print(f"Nb shape: {Nb.shape}")

# NOTE: matmul #2: (64,3)*(3,1)
out: NDArray[np.uint16] = Nb @ w_out // 256
print(out)
print(f"out shape: {out.shape}")


def tolabel(a):
    return 1 if a > 127 else 0


vectorized_tolabel = np.vectorize(tolabel)

esti = vectorized_tolabel(out)
# print(esti)

accuracy = np.equal(esti, labels).mean()
print(f"accuracy: {accuracy}")


X shape: (64, 7)
labels shape: (64,)
sigmoid shape: (256,)
w_hid shape: (8, 2)
w_out shape: (3,)
Xb shape: (64, 8)
[[ 46  20]
 [125  81]
 [111  84]
 [100  67]
 [ 80  56]
 [ 77  64]
 [111  84]
 [116  91]
 [115  76]
 [ 89  62]
 [ 75  54]
 [ 72  52]
 [126  94]
 [112  85]
 [114  84]
 [ 83  63]
 [130  95]
 [ 81  57]
 [107  82]
 [ 68  54]
 [ 84  62]
 [ 91  64]
 [115  84]
 [ 70  41]
 [105  79]
 [ 97  76]
 [114  89]
 [110  76]
 [ 75  52]
 [ 68  40]
 [ 70  53]
 [ 80  55]
 [ 71  58]
 [ 69  46]
 [105  74]
 [103  81]
 [ 81  53]
 [122  87]
 [ 71  46]
 [117  91]
 [ 70  45]
 [139 111]
 [ 66  51]
 [ 71  48]
 [ 82  65]
 [123  90]
 [136 102]
 [ 97  74]
 [113  85]
 [ 80  53]
 [ 65  40]
 [ 99  78]
 [103  75]
 [ 69  52]
 [108  85]
 [ 69  45]
 [ 63  39]
 [123  93]
 [ 80  50]
 [ 74  58]
 [113  88]
 [ 84  49]
 [ 96  68]
 [ 75  61]]
N shape: (64, 2)
Na shape: (64, 2)
Nb shape: (64, 3)
[100 152 151 134 122 127 151 159 146 128 120 118 165 153 152 127 167 123
 148 119 127 130 153 112 145 141 157 144 118 111 118 1

In [3]:
from pathlib import Path

def uint8_to_hex_word(k:int) -> str:
    k %= 256
    raw = hex(k)[2:].upper()
    return raw if len(raw) > 1 else '0'+raw

def store8bitmat(mat:NDArray[np.uint8], out_path: str) -> None:
  mat = mat.astype(np.uint8)
  flat = mat.ravel(order="C") # row major
  out_file = Path(out_path)
  with out_file.open("w") as f:
      for val in flat:
          f.write(uint8_to_hex_word(int(val)) + "\n")

input = np.concatenate((w_hid.flatten("F"),w_out.flatten(),X.flatten()))
print(f"input: {input.shape}")
# input: 8*2 + 3 + 64*7 = 467 (16+3+448) bytes

store8bitmat(input, "mlp_test_input.mem")

store8bitmat(out, "mlp_test_expected.mem")
# output: 64 bytes

input: (467,)


In [88]:
out

array([100, 152, 151, 134, 122, 127, 151, 159, 146, 128, 120, 118, 165,
       153, 152, 127, 167, 123, 148, 119, 127, 130, 153, 112, 145, 141,
       157, 144, 118, 111, 118, 122, 122, 114, 141, 146, 120, 157, 115,
       159, 114, 187, 117, 116, 128, 160, 177, 139, 153, 120, 110, 143,
       141, 117, 151, 114, 110, 164, 119, 122, 156, 119, 134, 125],
      dtype=uint16)